In [1]:
print("hi")

hi


```mermaid
graph LR
    A[CineBot Agent] -->|MCPAdapter| B[(MCP Server)]
    B --> C[check_showtimes]
    B --> D[cancel_booking]
    B --> E[get_seat_map]
```

In [2]:
import warnings,os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
assert os.environ.get("GROQ_API_KEY") is not None, "GROQ_API_KEY is not set in the environment variables."

In [4]:
!pip install -U "mcp<2" "langchain-mcp-adapters" langchain-groq
from langchain_mcp_adapters.client import MultiServerMCPClient

  Using cached mcp-1.30.0-py3-none-any.whl.metadata (9.3 kB)
Using cached mcp-1.30.0-py3-none-any.whl (234 kB)
  Attempting uninstall: mcp
    Found existing installation: mcp 2.3.0
    Uninstalling mcp-2.3.0:
      Successfully uninstalled mcp-2.3.0


In [5]:
# from langchain_mcp_adapters.client import MultiServerMCPClient
# print(MultiServerMCPClient.__name__)

# MCP Adapter in LangChain

In [6]:
# import langchain.mcp


In [7]:
%%writefile cinebot_mcp_server.py
from fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("CineBot")

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

@mcp.tool(annotations=ToolAnnotations(destructiveHint=True))
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

@mcp.tool()
def get_seat_map(movie_title: str) -> dict:
    """Get the seat map for a movie -- returns structured data, not just text."""
    return {"movie": movie_title, "available_rows": ["A", "B", "C"], "sold_out_rows": ["D"]}

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_mcp_server.py


In [8]:
!pip install "fastmcp-slim[client]"

  Using cached mcp-2.3.0-py3-none-any.whl.metadata (6.6 kB)
Using cached mcp-2.3.0-py3-none-any.whl (368 kB)
  Attempting uninstall: mcp
    Found existing installation: mcp 1.30.0
    Uninstalling mcp-1.30.0:
      Successfully uninstalled mcp-1.30.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-mcp-adapters 0.3.2 requires mcp<2.0.0,>=1.24.0, but you have mcp 2.3.0 which is incompatible.


In [9]:
%pip install "fastmcp-slim[client]==4.0.10" "fastmcp==4.0.10"

Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient
from fastmcp.client.transports import PythonStdioTransport

ImportError: FastMCP client support is not installed. Install `fastmcp` or `fastmcp-slim[client]`.

In [ ]:
cinebot_mcp_transport = PythonStdioTransport(
    script_path=Path("/Users/mayank/Github Repos/Agentic 3.0/Live-Class-2026/Weekend 14 - Oct 3 & Oct 4/MCP in Langchain/cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)

async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

NameError: name 'PythonStdioTransport' is not defined

In [ ]:
from langchain.agents import create_agent

# Call a tool without any AI or Agent


In [ ]:
tool_call_result=''
async with MCPAdapter(cinebot_transport) as adapter:
    tools = await adapter.list_tools()

    showtime_tool = next((t for t in tools if t.name == "check_showtimes"), None)

    direct_result = await showtime_tool.ainvoke({"movie_title": "interstellar"})
    print("Direct invocation result:", direct_result)

# Calling with Agent as well

    try:
        cinebot_agent = create_agent(model='groq:openai/gpt-oss-120b',tools=tools)
        result = await cinebot_agent.ainvoke({"messages":[('user', "What are the showtimes for Interstellar?")]})
        tool_call_result = result
        print("Agent invocation result:", result)
    except Exception as e:
        print("Error during agent invocation:", e)

Direct invocation result: [{'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_3852a2a6-2515-4130-9e62-9df2a749b46b'}]
Agent invocation result: {'messages': [HumanMessage(content='What are the showtimes for Interstellar?', additional_kwargs={}, response_metadata={}, id='a15714b5-9d35-4814-834b-c3eaa4fa5d09'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 194, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EUkUjDLPZqp0DPTdPmSHRLm4FERoq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id

In [ ]:
from rich import print as rprint
rprint("[bold green]Direct tool invocation result:[/bold green]", tool_call_result)

Direct tool invocation result:
{
    'messages': [
        HumanMessage(
            content='What are the showtimes for Interstellar?',
            additional_kwargs={},
            response_metadata={},
            id='a15714b5-9d35-4814-834b-c3eaa4fa5d09'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 26,
                    'prompt_tokens': 194,
                    'total_tokens': 220,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUkUjDLPZqp0DPTdPmSHRLm4FERoq',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0ffc1-95e1-7f53-a99f-dde045325988-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'movie_title': 'Interstellar'},
                    'id': 'call_2wcUW4Tso7jtZ0yrbjfZEfTW',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 194,
                'output_tokens': 26,
                'total_tokens': 220,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content=[
                {'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_f02f1757-50b6-436e-95d8-93686c1dcc53'}
            ],
            name='check_showtimes',
            id='7623b573-b8f6-437e-a3f3-0c981cee6b51',
            tool_call_id='call_2wcUW4Tso7jtZ0yrbjfZEfTW',
            artifact={'structured_content': {'result': '7:00 PM and 10:15 PM'}}
        ),
        AIMessage(
            content='Interstellar is showing at 7:00 PM and 10:15 PM. \n\nWould you like to book tickets or see the
seat map for either showing?',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 170,
                    'prompt_tokens': 236,
                    'total_tokens': 406,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 128,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUkUki86WrPYOJ7U0fqBX6HYJKTyf',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logp

In [ ]:
tool_call_result['messages'][-1]

AIMessage(content='Interstellar is showing at 7:00 PM and 10:15 PM. \n\nWould you like to book tickets or see the seat map for either showing?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 170, 'prompt_tokens': 236, 'total_tokens': 406, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EUkUki86WrPYOJ7U0fqBX6HYJKTyf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ffc1-9d48-7aa0-9fa6-c66837d8b725-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 236, 'output_tokens': 170, 'total_tokens': 406, 'input_token_details': {'audi

# 5. Tool Output: Multimodal Content Blocks

An MCP tool result isn't limited to text. `langchain.mcp` converts whatever the server sends —
text, images, embedded files — into standard LangChain **content blocks**, so a model sees a
uniform shape regardless of what kind of server produced it.

| MCP content type | Converts to |
|---|---|
| `TextContent` | text content block |
| `ImageContent` | image content block (base64 + mime type) |
| `ResourceLink` (image mime type) | image content block (by URL) |
| `ResourceLink` (other) | file content block |
| `EmbeddedResource` (text) | text content block |
| `EmbeddedResource` (blob) | image or file content block, by mime type |
| `AudioContent` | **not yet supported** — raises `NotImplementedError` |

# Structured Output 

In [ ]:
async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    seat_map_tool = next(t for t in tools if t.name == "get_seat_map")

    tool_call ={
        'name': 'get_seat_map',
        'args': {'movie_title': 'Interstellar'},
        'id': 'unique_call_id_12345',  # Optional: Provide a unique ID for the call
        'type':'tool_call'
    }

    message = await seat_map_tool.ainvoke(tool_call)
    print("Text Content (what my model reads)",message.content)
    print("Structured Data (what my code can use if required)",message.artifact)
    print(message.artifact['structured_content']['available_rows'])  # Accessing structured data directly

Text Content (what my model reads) [{'type': 'text', 'text': '{"movie":"Interstellar","available_rows":["A","B","C"],"sold_out_rows":["D"]}', 'id': 'lc_b4f191a2-de81-4303-a2fb-c47c3cc851d9'}]
Structured Data (what my code can use if required) {'structured_content': {'movie': 'Interstellar', 'available_rows': ['A', 'B', 'C'], 'sold_out_rows': ['D']}}
['A', 'B', 'C']


# Error handling



| Failure | What happened | What the agent sees |
|---|---|---|
| Server tool **ran and reported failure** (`isError=True`) | The tool executed, then explicitly failed (bad input, business-rule violation) | A `ToolMessage` with `status="error"` — **the agent can read it and self-correct**, same as any other tool error in this course |
| **Transport / session failure** | The connection dropped, the server crashed, the process couldn't start | **Raises an exception** — there is no error `ToolMessage` for the model to reason about, because nothing came back to convert |


In [ ]:
%%writefile cinebot_error_server.py
from fastmcp import FastMCP
mcp = FastMCP("CineBot")

@mcp.tool()
def risky_lookup(booking_id: str) -> str:
    """Look up a booking -- fails if the ID format is wrong."""
    if not booking_id.startswith("BK"):
        raise ValueError(f"Invalid booking ID format: {booking_id!r}. Expected it to start with 'BK'.")
    return f"Booking {booking_id}: confirmed."

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_error_server.py


In [ ]:
error_message_global = ''
error_transport = PythonStdioTransport(
    script_path=Path('cinebot_error_server.py'),
    log_file=Path('cinebot_error_server.log')
)

async with MCPAdapter(error_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    risky_lookup_tool = tools[0]

    bad_call = {'name': 'risky_lookup', 'args': {'booking_id': '12345'}, 'id': 'bad_call_001', 'type': 'tool_call'}

    error_message = await risky_lookup_tool.ainvoke(bad_call)
    print("Error message from tool:", error_message)
    print("Content:", error_message.content)
    error_message_global = error_message
    print(error_message)
  

Error message from tool: content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_6fbaf2a7-5d9c-4ecf-92ac-6023ecacb373'}] name='risky_lookup' tool_call_id='bad_call_001' status='error'
Content: [{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_6fbaf2a7-5d9c-4ecf-92ac-6023ecacb373'}]
content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_6fbaf2a7-5d9c-4ecf-92ac-6023ecacb373'}] name='risky_lookup' tool_call_id='bad_call_001' status='error'


In [ ]:
error_message_global

ToolMessage(content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_6fbaf2a7-5d9c-4ecf-92ac-6023ecacb373'}], name='risky_lookup', tool_call_id='bad_call_001', status='error')

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware



In [ ]:
def is_destructive_tool_call(tool):
    # return message.type == "tool_call" and message.name == "cancel_booking"
    meta = (tool.metadata or {}).get('mcp',{}).get('tool',{}).get('annotations',{})
    return bool(meta.get('destructive_hint'))


async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    for t in tools:
        print(f"Tool: {t.name}, Destructive: {is_destructive_tool_call(t)}")


Tool: check_showtimes, Destructive: False
Tool: cancel_booking, Destructive: True
Tool: get_seat_map, Destructive: False


In [ ]:
Tool: cancel_booking, Destructive: False, Tools:name='cancel_booking' description='Cancel an existing booking. Irreversible.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'booking_id': {'type': 'string'}}, 'required': ['booking_id']} metadata={'mcp': {'tool': {'annotations': {'destructive_hint': True}, '_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}} handle_tool_error=<function _handle_mcp_tool_error at 0x1163c74c0> response_format='content_and_artifact' coroutine=<function as_langchain_tool.<locals>.call_tool at 0x12287c7c0>


In [ ]:
async with MCPAdapter(cinebot_transport) as adapter:
    tools = await adapter.list_tools()

    for t in tools:
        print(f"{t.name}: destructive={is_destructive_tool_call(t)}")

    # One HITL config, built dynamically from whatever the server marked destructive.
    interrupt_on = {
        t.name: {"allowed_decisions": ["approve", "edit", "reject"]}
        for t in tools
        if is_destructive_tool_call(t)
    }
    print("\ninterrupt_on config:", interrupt_on)

    hitl_middleware = HumanInTheLoopMiddleware(interrupt_on=interrupt_on)
    cinebot_guarded = create_agent(model="groq:openai/gpt-oss-120b", tools=tools, middleware=[hitl_middleware])

check_showtimes: destructive=False
cancel_booking: destructive=True
get_seat_map: destructive=False

interrupt_on config: {'cancel_booking': {'allowed_decisions': ['approve', 'edit', 'reject']}}


---
# 9. Elicitation: When the Server Needs to Ask a Question Mid-Call

Some MCP tools need clarification partway through a call — "confirm you want to proceed?".
`langchain.mcp` answers this **automatically with a LangGraph `interrupt()`** — every client
`MCPAdapter` builds is armed for it by default, no extra code needed to enable it.

```mermaid
sequenceDiagram
    participant Agent
    participant MCPAdapter
    participant Server
    Agent->>MCPAdapter: call cancel_booking(...)
    MCPAdapter->>Server: tools/call
    Server-->>MCPAdapter: needs more input (elicitation)
    MCPAdapter->>Agent: interrupt() -- pauses the graph
    Note over Agent: A human answers, outside the graph
    Agent->>MCPAdapter: Command(resume={"responses": {key: answer}})
    MCPAdapter->>Server: tools/call, retried with the answer
    Server-->>MCPAdapter: final result
```
